## UCS619 Quantum Computing: Lab Assignment-10
## QFT, QPE, and Integer Factorization

Name: Smarth Kaushal | Roll No. 102497023 | Group: 3Q21 | Lab Instructor: Dr. Samya Muhuri

---
## Experiment 1: Quantum Fourier Transform (QFT)

In [1]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

from qiskit import transpile
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram
from qiskit.circuit.library import QFT

print("All imports successful!")

All imports successful!


In [2]:
def qft_rotations(circuit, n):
    """Recursively apply QFT rotations on the first n qubits.
    For each qubit: apply Hadamard, then controlled-phase rotations.
    """
    if n == 0:
        return circuit
    n -= 1
    circuit.h(n)
    for qubit in range(n):
        circuit.cp(np.pi / 2 ** (n - qubit), qubit, n)
        
    return qft_rotations(circuit, n)


def swap_registers(circuit, n):
    """Swap qubits to restore correct bit order after QFT."""
    for qubit in range(n // 2):
        circuit.swap(qubit, n - qubit - 1)
    return circuit


def apply_qft(circuit, n):
    """Full QFT on first n qubits: rotations then bit-reversal swap."""
    qft_rotations(circuit, n)
    swap_registers(circuit, n)
    return circuit


print("QFT functions defined.")

QFT functions defined.


In [8]:
# Build QFT circuit on input state |5> = |101>
n_qubits = 3

qc = QuantumCircuit(n_qubits)
qc.x(0)   # q0 = 1
qc.x(2)   # q2 = 1  ->  |101> = 5 (little-endian)
qc.barrier()
apply_qft(qc, n_qubits)

print("QFT Circuit on |5> = |101>:")

try:
    fig = qc.draw('mpl')
    plt.savefig('images/qft_circuit.png', bbox_inches='tight', dpi=100)
    plt.show()
    plt.close()
except Exception:
    print(qc.draw('text'))

QFT Circuit on |5> = |101>:


C:\Users\Smarth Kaushal\AppData\Local\Temp\ipykernel_5772\1461253117.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# Statevector simulation - print exact amplitudes
sv = Statevector.from_instruction(qc)

print(f"{'State':>8}  {'Amplitude':>42}  {'Probability':>12}")
print("-" * 68)
for i, amp in enumerate(sv):
    label = "|" + format(i, f'0{n_qubits}b') + ">"
    print(f"{label:>8}  {str(amp):>42}  {abs(amp)**2:>12.6f}")

   State                                   Amplitude   Probability
--------------------------------------------------------------------
   |000>                     (0.3535533905932737+0j)      0.125000
   |001>  (-0.24999999999999994-0.24999999999999994j)      0.125000
   |010>  (2.1648901405887326e-17+0.3535533905932737j)      0.125000
   |011>  (0.24999999999999994-0.24999999999999994j)      0.125000
   |100>                    (-0.3535533905932737+0j)      0.125000
   |101>  (0.24999999999999994+0.24999999999999994j)      0.125000
   |110>  (-2.1648901405887326e-17-0.3535533905932737j)      0.125000
   |111>  (-0.24999999999999994+0.24999999999999994j)      0.125000


In [9]:
# Shots-based measurement - all 8 states should appear with equal probability (~1/8)
qc_meas = QuantumCircuit(n_qubits, n_qubits)
qc_meas.x(0)
qc_meas.x(2)
qc_meas.barrier()
apply_qft(qc_meas, n_qubits)
qc_meas.measure(range(n_qubits), range(n_qubits))

sim = AerSimulator()
counts = sim.run(qc_meas, shots=1024).result().get_counts()

print("Measurement counts:", counts)
fig = plot_histogram(counts, title='QFT Measurement on |5>')
plt.savefig('images/qft_measurement.png', bbox_inches='tight', dpi=100)
plt.show()
plt.close()

Measurement counts: {'000': 119, '010': 133, '001': 144, '101': 120, '011': 124, '111': 129, '110': 130, '100': 125}


C:\Users\Smarth Kaushal\AppData\Local\Temp\ipykernel_5772\2914618905.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# Inverse QFT demo - apply QFT then IQFT; must recover |101> with certainty.
def apply_inverse_qft(circuit, n):
    """Append the inverse QFT (IQFT) to 'circuit' on its first n qubits."""
    qft_circ  = apply_qft(QuantumCircuit(n), n)
    # FIX 6: The original code passed iqft_circ (a QuantumCircuit) directly
    # to circuit.append(), which requires an Instruction/Gate, not a plain
    # QuantumCircuit.  Adding .to_gate() converts it to the correct type.
    iqft_gate = qft_circ.inverse().to_gate(label='IQFT')
    circuit.append(iqft_gate, range(n))
    return circuit


qc_inv = QuantumCircuit(n_qubits, n_qubits)
qc_inv.x(0)
qc_inv.x(2)
qc_inv.barrier()
apply_qft(qc_inv, n_qubits)
apply_inverse_qft(qc_inv, n_qubits)
qc_inv.measure(range(n_qubits), range(n_qubits))

compiled_qc_inv = transpile(qc_inv, sim)
counts_inv = sim.run(compiled_qc_inv, shots=1024).result().get_counts()
print("After QFT then IQFT (should be '101' only):", counts_inv)

After QFT then IQFT (should be '101' only): {'101': 1024}


---
## Experiment 2: Quantum Phase Estimation (QPE)

In [11]:
from qiskit import QuantumRegister, ClassicalRegister
from fractions import Fraction

n_counting = 4   # 4 counting qubits -> precision 1/16

print(f"Counting qubits : {n_counting}")
print(f"Phase precision : 1/{2**n_counting} = {1/2**n_counting}")

Counting qubits : 4
Phase precision : 1/16 = 0.0625


In [12]:
def build_qpe_circuit(unitary_qc, eigenstate_qc, n_counting):
    """Build the standard QPE circuit.

    Steps:
    1. Prepare eigenstate in the target register.
    2. Apply Hadamard to every counting qubit.
    3. Apply controlled-U^(2^j) for each counting qubit j.
    4. Apply inverse QFT to the counting register.
    5. Measure the counting register.
    """
    n_target  = eigenstate_qc.num_qubits
    qr_count  = QuantumRegister(n_counting, name='count')
    qr_target = QuantumRegister(n_target,   name='target')
    cr        = ClassicalRegister(n_counting, name='result')
    qc = QuantumCircuit(qr_count, qr_target, cr)

    # Step 1 - eigenstate preparation
    qc.append(eigenstate_qc.to_gate(label=eigenstate_qc.name), qr_target)
    qc.barrier()

    # Step 2 - Hadamard on counting register
    for q in range(n_counting):
        qc.h(q)
    qc.barrier()

    # Step 3 - controlled-U^(2^j) for each counting qubit
    repetitions = 1
    for j in range(n_counting):
        ctrl_u = unitary_qc.to_gate(label=unitary_qc.name).control(1)
        for _ in range(repetitions):
            qc.append(ctrl_u, [qr_count[j]] + list(qr_target))
        repetitions *= 2
    qc.barrier()

    # Step 4 - inverse QFT on counting register
    # do_swaps=True keeps the bit-reversal so measured bits are in standard binary order (MSB first) for direct phase reading.
    iqft_gate = QFT(n_counting, inverse=True, do_swaps=True).to_gate(label='IQFT')
    qc.append(iqft_gate, qr_count)
    qc.barrier()

    # Step 5 - measure
    qc.measure(qr_count, cr)
    return qc


print("QPE circuit builder defined.")

QPE circuit builder defined.


In [44]:
# QPE for the T gate - true phase theta = 1/8 = 0.125
# T|1> = e^{i*pi/4}|1>  ->  2*pi*theta = pi/4  ->  theta = 1/8

u_T   = QuantumCircuit(1, name='T');  u_T.t(0)
# FIX 9: Renamed eigenstate circuit from '|1>' to 'e1' to avoid special
# characters in the gate label that can break some Qiskit backends.
eig_T = QuantumCircuit(1, name='e1'); eig_T.x(0)

qpe_T    = build_qpe_circuit(u_T, eig_T, n_counting)
compiled_qpe_T = transpile(qpe_T, sim)
counts_T = sim.run(compiled_qpe_T, shots=2048).result().get_counts()

best_T = max(counts_T, key=counts_T.get)
est_T  = int(best_T, 2) / (2 ** n_counting)

print("T gate - QPE Results")
print(f"Most frequent outcome : |{best_T}> = {int(best_T, 2)}")
print(f"Estimated phase       : {int(best_T, 2)}/{2**n_counting} = {est_T}")
print(f"True phase            : {1/8}")
print(f"Error                 : {abs(est_T - 1/8):.6f}")

fig = plot_histogram(counts_T, title='QPE: T gate (theta = 1/8)')
plt.savefig('qpe_T.png', bbox_inches='tight', dpi=100)
plt.show()
plt.close()

C:\Users\A\AppData\Local\Temp\ipykernel_5204\830924801.py:44: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  iqft_gate = QFT(n_counting, inverse=True, do_swaps=True).to_gate(label='IQFT')


T gate - QPE Results
Most frequent outcome : |0010> = 2
Estimated phase       : 2/16 = 0.125
True phase            : 0.125
Error                 : 0.000000


C:\Users\A\AppData\Local\Temp\ipykernel_5204\44935385.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
# QPE for the S gate - true phase theta = 1/4 = 0.25
# S|1> = e^{i*pi/2}|1>  ->  2*pi*theta = pi/2  ->  theta = 1/4

u_S   = QuantumCircuit(1, name='S');  u_S.s(0)
eig_S = QuantumCircuit(1, name='e1'); eig_S.x(0)

qpe_S    = build_qpe_circuit(u_S, eig_S, n_counting)
compiled_qpe_S = transpile(qpe_S, sim)
counts_S = sim.run(compiled_qpe_S, shots=2048).result().get_counts()

best_S = max(counts_S, key=counts_S.get)
est_S  = int(best_S, 2) / (2 ** n_counting)

print("S gate - QPE Results")
print(f"Most frequent outcome : |{best_S}> = {int(best_S, 2)}")
print(f"Estimated phase       : {int(best_S, 2)}/{2**n_counting} = {est_S}")
print(f"True phase            : {1/4}")
print(f"Error                 : {abs(est_S - 1/4):.6f}")

fig = plot_histogram(counts_S, title='QPE: S gate (theta = 1/4)')
plt.savefig('images/qpe_S.png', bbox_inches='tight', dpi=100)
plt.show()
plt.close()

C:\Users\Smarth Kaushal\AppData\Local\Temp\ipykernel_5772\2002749128.py:37: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  iqft_gate = QFT(n_counting, inverse=True, do_swaps=True).to_gate(label='IQFT')


S gate - QPE Results
Most frequent outcome : |0100> = 4
Estimated phase       : 4/16 = 0.25
True phase            : 0.25
Error                 : 0.000000


C:\Users\Smarth Kaushal\AppData\Local\Temp\ipykernel_5772\3644340147.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
# QPE for the Z gate - true phase theta = 1/2 = 0.5
# Z|1> = e^{i*pi}|1>  ->  2*pi*theta = pi  ->  theta = 1/2

u_Z   = QuantumCircuit(1, name='Z');  u_Z.z(0)
eig_Z = QuantumCircuit(1, name='e1'); eig_Z.x(0)

qpe_Z    = build_qpe_circuit(u_Z, eig_Z, n_counting)

# FIXED: Transpile the qpe_Z circuit before running
compiled_qpe_Z = transpile(qpe_Z, sim)

# FIXED: Use the compiled circuit in sim.run()
counts_Z = sim.run(compiled_qpe_Z, shots=2048).result().get_counts()

best_Z = max(counts_Z, key=counts_Z.get)
est_Z  = int(best_Z, 2) / (2 ** n_counting)

print("Z gate - QPE Results")
print(f"Most frequent outcome : |{best_Z}> = {int(best_Z, 2)}")
print(f"Estimated phase       : {int(best_Z, 2)}/{2**n_counting} = {est_Z}")
print(f"True phase            : {1/2}")
print(f"Error                 : {abs(est_Z - 1/2):.6f}")

fig = plot_histogram(counts_Z, title='QPE: Z gate (theta = 1/2)')
plt.savefig('images/qpe_Z.png', bbox_inches='tight', dpi=100)
plt.show()
plt.close()

C:\Users\Smarth Kaushal\AppData\Local\Temp\ipykernel_5772\2002749128.py:37: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  iqft_gate = QFT(n_counting, inverse=True, do_swaps=True).to_gate(label='IQFT')


Z gate - QPE Results
Most frequent outcome : |1000> = 8
Estimated phase       : 8/16 = 0.5
True phase            : 0.5
Error                 : 0.000000


C:\Users\Smarth Kaushal\AppData\Local\Temp\ipykernel_5772\3099183074.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Experiment 3: Shor's Integer Factorization

We factor $N = 15$ using Shor's algorithm.  
- **Quantum Part:** Uses Quantum Phase Estimation (QPE) to find the period $r$ of the modular function $f(x) = a^x \pmod N$.  
- **Classical Post-processing:** Recovers the factors of $N$ by calculating $\gcd(a^{r/2} \pm 1, N)$.

In [17]:
import math

N = 15

# Classical pre-checks
print(f"Factoring N = {N}")
print(f"Is N even?                     : {N % 2 == 0}")
print(f"gcd(2,15)={math.gcd(2,15)},  gcd(3,15)={math.gcd(3,15)},  gcd(5,15)={math.gcd(5,15)}")

valid_bases = [a for a in [2, 4, 7, 8, 11, 13] if math.gcd(a, N) == 1]
print(f"Valid coprime bases for Shor's : {valid_bases}")

Factoring N = 15
Is N even?                     : False
gcd(2,15)=1,  gcd(3,15)=3,  gcd(5,15)=5
Valid coprime bases for Shor's : [2, 4, 7, 8, 11, 13]


In [18]:
def c_amod15(a, power):
    """Return a controlled-U gate implementing U|y> = |a*y mod 15>.

    Simplified hard-coded swap network for N = 15.
    Valid bases: a in {2, 4, 7, 8, 11, 13}.
    """
    U = QuantumCircuit(4, name=f"{a}^{power} mod15")
    for _ in range(power):
        if a in [2, 13]:
            U.swap(0, 1); U.swap(1, 2); U.swap(2, 3)
        elif a in [7, 8]:
            U.swap(2, 3); U.swap(1, 2); U.swap(0, 1)
        elif a in [4, 11]:
            U.swap(1, 3); U.swap(0, 2)
    return U.to_gate().control(1)


def build_qft_dagger(n):
    """Return the inverse QFT (QFT-dagger) as a QuantumCircuit on n qubits.

    Bug A (gate ordering): The previous code applied H BEFORE the controlled-
    phase (CP) rotations.  This is the FORWARD QFT order, not its inverse.
    In the correct IQFT the CP corrections must come FIRST, then H.

    Bug B (swap position): The bit-reversal SWAPs of the QFT must be undone
    FIRST (at the start of the IQFT), because they were the LAST operation in
    the forward QFT.  The previous code placed the SWAPs at the END, which
    produced a completely wrong unitary.

    Correct IQFT gate sequence for n qubits:
      Step 1.  SWAP(i, n-1-i)  for i = 0 ... n//2-1   <- undo QFT swaps
      Step 2.  For j = 0, 1, ..., n-1:
                   CP(-pi/2^(j-m), m, j)  for m = 0 ... j-1
                   H(j)                               <- H comes AFTER CPs
    """
    qc = QuantumCircuit(n, name='QFT_dagger')

    # Step 1: undo the final bit-reversal swaps of the forward QFT
    for i in range(n // 2):
        qc.swap(i, n - i - 1)

    # Step 2: undo rotation-and-Hadamard layers in forward qubit order
    for j in range(n):
        # CP corrections with negated angles (undoes forward QFT CP gates)
        for m in range(j):
            qc.cp(-np.pi / float(2 ** (j - m)), m, j)
        # H comes AFTER the CP corrections (inverse of: H then CP)
        qc.h(j)

    return qc


def build_shor_circuit(a, N=15, n_count=8):
    """Build Shor's QPE circuit for modular base a and target N = 15."""
    qc = QuantumCircuit(n_count + 4, n_count)
    qc.x(n_count)          # initialise work register to |1>
    for q in range(n_count):
        qc.h(q)            # Hadamard on all counting qubits
    qc.barrier()

    # Controlled-U^(2^q) gates
    for q in range(n_count):
        qc.append(c_amod15(a, 2 ** q),
                  [q] + list(range(n_count, n_count + 4)))
    qc.barrier()

    qc.append(build_qft_dagger(n_count).to_gate(), range(n_count))
    qc.barrier()

    qc.measure(range(n_count), range(n_count))
    return qc


print("Shor's circuit functions defined.")

Shor's circuit functions defined.


In [20]:
# Run Shor's algorithm for every valid coprime base
n_count     = 8
all_factors = set()

for a in valid_bases:
    print(f"\n--- Base a = {a} ---")
    qc_shor  = build_shor_circuit(a, N, n_count)
    
    # FIXED: Transpile the circuit before running!
    compiled_shor = transpile(qc_shor, sim)
    
    # FIXED: Run the compiled circuit
    counts_s = sim.run(compiled_shor, shots=2048).result().get_counts()

    # Extract period candidates via continued-fraction convergents
    factors = set()
    for outcome in sorted(counts_s, key=lambda x: -counts_s[x])[:8]:
        y     = int(outcome, 2)
        phase = y / (2 ** n_count)
        if phase == 0:
            continue
        r = Fraction(phase).limit_denominator(N).denominator
        if r % 2 != 0:
            continue
        ar2 = pow(a, r // 2, N)
        if ar2 == N - 1:
            continue
        for f in [math.gcd(ar2 + 1, N), math.gcd(ar2 - 1, N)]:
            if 1 < f < N and N % f == 0:
                factors.add(f)

    if factors:
        print(f"  Factors found : {factors}")
        all_factors.update(factors)
    else:
        print("  No factors this run (algorithm is probabilistic - re-run if needed).")


--- Base a = 2 ---
  Factors found : {3, 5}

--- Base a = 4 ---
  Factors found : {3, 5}

--- Base a = 7 ---
  Factors found : {3, 5}

--- Base a = 8 ---
  Factors found : {3, 5}

--- Base a = 11 ---
  Factors found : {3, 5}

--- Base a = 13 ---
  Factors found : {3, 5}


In [21]:
# Print the final factorisation result
print(f"\nFactoring N = {N}")
found = False
for p in sorted(all_factors):
    if 1 < p < N and N % p == 0:
        q_factor = N // p
        if q_factor > 1:
            print(f"\n  Result : {N} = {p} x {q_factor}")
            print(f"  Verify : {p} x {q_factor} = {p * q_factor}  ->  {'CORRECT' if p * q_factor == N else 'ERROR'}")
            found = True
            break
if not found:
    print("  No non-trivial factors found in this run. Re-run (probabilistic algorithm).")


Factoring N = 15

  Result : 15 = 3 x 5
  Verify : 3 x 5 = 15  ->  CORRECT


In [22]:
# Visualise: QPE histogram for a=2, plus a compact 4-qubit circuit diagram
qc_a2     = build_shor_circuit(2, N, n_count)

# FIXED: Transpile the circuit before running
compiled_qc_a2 = transpile(qc_a2, sim)

# FIXED: Run the compiled circuit
counts_a2 = sim.run(compiled_qc_a2, shots=2048).result().get_counts()

print("Shor's QPE Circuit (a=2, N=15, n_count=4 for readability):")
try:
    fig = build_shor_circuit(2, N, n_count=4).draw('mpl', fold=80)
    plt.savefig('shor_circuit.png', bbox_inches='tight', dpi=100)
    plt.show()
    plt.close()
except Exception:
    print(build_shor_circuit(2, N, n_count=4).draw('text'))

print("\nShor's QPE measurement histogram (a=2, N=15, n_count=8):")
fig2 = plot_histogram(counts_a2, title="Shor's QPE: a=2, N=15", figsize=(12, 4))
plt.savefig('images/shor_histogram.png', bbox_inches='tight', dpi=100)
plt.show()
plt.close()

Shor's QPE Circuit (a=2, N=15, n_count=4 for readability):


C:\Users\Smarth Kaushal\AppData\Local\Temp\ipykernel_5772\191805471.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



Shor's QPE measurement histogram (a=2, N=15, n_count=8):


C:\Users\Smarth Kaushal\AppData\Local\Temp\ipykernel_5772\191805471.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
